In [29]:
marcas = ['Toyota', 'Honda', 'Ford', 'Chevrolet', 'Nissan', 'Mazda', 'BMW', 'Audi']

vehiculos = []
for i in range(3):
    vehiculo = {
        'id_vehiculo': f"{random.choice(marcas)}_{i+1}",
        'millas_por_galon_mpg': round(np.random.uniform(15, 35), 1),
        'cilindros_motor_cyl': random.choice([4, 6, 8]),
        'caballos_fuerza_hp': random.randint(100, 350),
        'peso_wt': round(np.random.uniform(2.0, 5.0), 2)
    }
    vehiculos.append(vehiculo)

vehiculos

[{'id_vehiculo': 'Ford_1',
  'millas_por_galon_mpg': 19.8,
  'cilindros_motor_cyl': 6,
  'caballos_fuerza_hp': 336,
  'peso_wt': 3.31},
 {'id_vehiculo': 'Chevrolet_2',
  'millas_por_galon_mpg': 32.7,
  'cilindros_motor_cyl': 8,
  'caballos_fuerza_hp': 275,
  'peso_wt': 2.87},
 {'id_vehiculo': 'Honda_3',
  'millas_por_galon_mpg': 30.7,
  'cilindros_motor_cyl': 4,
  'caballos_fuerza_hp': 340,
  'peso_wt': 4.28}]

In [1]:
import numpy as np
from datetime import datetime, timedelta
import random

# Configuración de semilla para reproducibilidad
np.random.seed(2024)
random.seed(2024)

# Generación de datos simulados para múltiples sesiones de prueba
def generar_sesion_prueba(id_sesion: str, n_vehiculos: int) -> dict:
    """Genera una sesión completa con múltiples ensayos de vehículos."""
    marcas = ['Toyota', 'Honda', 'Ford', 'Chevrolet', 'Nissan', 'Mazda', 'BMW', 'Audi']
    modelos = ['Sedan', 'SUV', 'Coupe', 'Hatchback']
    
    vehiculos = []
    for i in range(n_vehiculos):
        vehiculo = {
            'id_vehiculo': f"{random.choice(marcas)}_{i+1}",
            'millas_por_galon_mpg': round(np.random.uniform(15, 35), 1),
            'cilindros_motor_cyl': random.choice([4, 6, 8]),
            'caballos_fuerza_hp': random.randint(100, 350),
            'peso_wt': round(np.random.uniform(2.0, 5.0), 2)
        }
        vehiculos.append(vehiculo)
    
    return {
        'id_sesion_laboratorio': id_sesion,
        'marca_tiempo_prueba': (datetime.now() - timedelta(days=random.randint(0, 30))).isoformat(),
        'investigador_responsable': random.choice(['Dr. Fisher', 'Dra. Pearson', 'Dr. Tukey']),
        'ensayos_experimentales': vehiculos
    }

# Generar 3 sesiones con diferentes números de vehículos
sesiones_simuladas = [
    generar_sesion_prueba('SES-2024-AA', 5),
    generar_sesion_prueba('SES-2024-BB', 3),  # Mínimo permitido
    generar_sesion_prueba('SES-2024-CC', 2),  # Debería fallar por tamaño insuficiente
]

print(f"Generadas {len(sesiones_simuladas)} sesiones de prueba simuladas")
for s in sesiones_simuladas:
    print(f"  {s['id_sesion_laboratorio']}: {len(s['ensayos_experimentales'])} vehículos")


Generadas 3 sesiones de prueba simuladas
  SES-2024-AA: 5 vehículos
  SES-2024-BB: 3 vehículos
  SES-2024-CC: 2 vehículos


In [30]:
from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import List, Dict, Any
from datetime import datetime

# 1. NIVEL MICRO: Sub-modelo dependiente (Unidad observacional)
class EnsayoVehiculo(BaseModel):
    """
    Modelo para una unidad observacional individual (Nivel 1).
    Equivalente al sujeto muestral base en un diseño jerárquico.
    """
    id_vehiculo: str = Field(
        min_length=2, 
        description="Nomenclatura única del automotor"
    )
    millas_por_galon_mpg: float = Field(
        ..., 
        gt=0.0,
        description="Eficiencia termodinámica del combustible"
    )
    cilindros_motor_cyl: int = Field(
        ..., 
        description="Topología geométrica de la cámara de combustión"
    )
    caballos_fuerza_hp: int = Field(
        ..., 
        gt=0,
        description="Potencia bruta del motor"
    )
    peso_wt: float = Field(
        ..., 
        gt=0.0, le=10.0,
        description="Peso del vehículo en miles de libras"
    )
    
    @field_validator('cilindros_motor_cyl')
    @classmethod
    def auditar_arquitectura_motor(cls, v: int) -> int:
        """
        Filtro axiomático: Solo permite arquitecturas de motores
        de combustión interna empíricamente fabricadas.
        """
        arquitecturas_validas = {4, 6, 8}
        if v not in arquitecturas_validas:
            raise ValueError(
                f"Fallo Paramétrico. Arquitectura geométrica atípica: "
                f"{v} cilindros. Valores permitidos: {arquitecturas_validas}"
            )
        return v

# 2. NIVEL MACRO: Modelo Principal Contenedor (Bloque experimental)
class SesionDinamometro(BaseModel):
    """
    Modelo para el bloque experimental completo (Nivel 2).
    Representa la parcela entera en un diseño de parcelas divididas.
    """
    # Identificador con patrón estandarizado riguroso
    id_sesion_laboratorio: str = Field(
        pattern=r'^SES-\d{4}-[A-Z]{2}$',
        description="Formato: SES-AAAA-XX"
    )
    
    marca_tiempo_prueba: datetime = Field(
        default_factory=datetime.utcnow,
        description="Sello temporal GMT de calibración"
    )
    
    investigador_responsable: str = Field(
        min_length=3,
        description="Nombre del investigador a cargo"
    )
    
    # ESTRUCTURA DE ANIDAMIENTO CORE:
# Lista tipada que alberga múltiples objetos de la clase dependiente
    ensayos_experimentales: List[EnsayoVehiculo]
    
    @field_validator('ensayos_experimentales')
    @classmethod
    def validar_potencia_muestral_del_bloque(
        cls, 
        ensayos_vector: List[EnsayoVehiculo]
    ) -> List[EnsayoVehiculo]:
        """
        Comprobación estadística macroscópica de los requerimientos
        de tamaño de muestra (Power Analysis) a nivel del bloque completo.
        
        Requiere mínimo 3 unidades para estimar varianza residual
        intra-bloque con fiabilidad asintótica.
        """
        if len(ensayos_vector) < 3:
            raise ValueError(
                f"Insuficiencia de Grados de Libertad: El diseño estadístico "
                f"exige al menos 3 vehículos por sesión. Recibidos: {len(ensayos_vector)}"
            )
        return ensayos_vector

# 3. Función para procesar sesiones con manejo de errores jerárquico
def procesar_sesion_experimento(datos_crudos: Dict[str, Any]) -> dict:
    """
    Procesa una sesión experimental completa, validando tanto
    el nivel macro (sesión) como el nivel micro (vehículos anidados).
    """
    try:
        sesion_certificada = SesionDinamometro.model_validate(datos_crudos)
        return {
            'estado': 'VALIDADO',
            'sesion': sesion_certificada,
            'n_vehiculos': len(sesion_certificada.ensayos_experimentales)
        }
    except ValidationError as e:
        errores = e.errors()
        return {
            'estado': 'RECHAZADO',
            'errores': [
                {
                    'campo': ' -> '.join(str(x) for x in err['loc']),
                    'mensaje': err['msg'],
                    'tipo': err['type']
                }
                for err in errores
            ]
        }

# 4. Procesamiento de las sesiones simuladas
print("\n" + "="*60)
print("RESULTADOS DE VALIDACIÓN JERÁRQUICA")
print("="*60)

for sesion_data in sesiones_simuladas:
    resultado = procesar_sesion_experimento(sesion_data)
    print(f"\n📋 Sesión: {sesion_data['id_sesion_laboratorio']}")
    
    if resultado['estado'] == 'VALIDADO':
        print(f"   ✅ VALIDADA")
        print(f"   Vehículos procesados: {resultado['n_vehiculos']}")
    else:
        print(f"   ❌ RECHAZADA")
        for error in resultado['errores']:
            print(f"   • Campo: {error['campo']}")
            print(f"     Error: {error['mensaje'][:60]}...")



RESULTADOS DE VALIDACIÓN JERÁRQUICA

📋 Sesión: SES-2024-AA
   ✅ VALIDADA
   Vehículos procesados: 5

📋 Sesión: SES-2024-BB
   ✅ VALIDADA
   Vehículos procesados: 3

📋 Sesión: SES-2024-CC
   ❌ RECHAZADA
   • Campo: ensayos_experimentales
     Error: Value error, Insuficiencia de Grados de Libertad: El diseño ...
